#  TRABAJO FINAL LP2

Hecha por:
-  Pariona Huarhauchi Misael Gabriel - 20240725
-  Marca Andia Hernan Joel - 20231497
-  Chamilco Carhuamaca Diego - 20240702

ADVERTENCIA

Antes de ejecutra los codigos, se recomienda leer las indicaciones.


PAQUETES


In [13]:
pip install selenium pandas webdriver-manager

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
pip install --upgrade webdriver-manager selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
%pip install -q -U google-generativeai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
%pip install webdriver-manager

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


PRIMER BLOQUE DE CODIGO


In [ ]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import NoSuchElementException, TimeoutException

def configurar_driver():
    """Configura el navegador simulando ser un usuario humano."""
    opciones = Options()
    opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36")
    opciones.add_argument("--disable-blink-features=AutomationControlled")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    
    servicio = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=servicio, options=opciones)
    driver.set_page_load_timeout(20) 
    return driver

def obtener_enlaces_ebay(driver, cantidad_objetivo=50):
    """FASE 1: Extrae los enlaces específicos del iPhone 17 Pro Max."""
    url_busqueda = "https://www.ebay.com/sch/i.html?_nkw=iphone+17+pro+max"
    print(f"Abriendo la página de listado: {url_busqueda}")
    
    try:
        driver.get(url_busqueda)
    except TimeoutException:
        driver.execute_script("window.stop();")
    
    # Pausa manual para evadir el Firewall (¡Como te funcionó la última vez!)
    print("\n" + "="*60)
    print("PAUSA DE SEGURIDAD ACTIVADA")
    print("1. Ve a la ventana de Chrome que se abrió.")
    print("2. Si eBay te pide verificar que eres humano, resuélvelo.")
    print("3. Asegúrate de estar viendo los iPhone 17 Pro Max en pantalla.")
    print("4. Vuelve a esta consola negra...")
    input("PRESIONA LA TECLA 'ENTER' AQUÍ PARA QUE EL BOT ARRANQUE: ")
    print("="*60 + "\n")
    
    enlaces = set() 
    
    while len(enlaces) < cantidad_objetivo:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        elementos_a = driver.find_elements(By.TAG_NAME, "a")
        
        for a in elementos_a:
            href = a.get_attribute("href")
            if href and "/itm/" in href:
                link_limpio = href.split("?")[0]
                enlaces.add(link_limpio)
                if len(enlaces) >= cantidad_objetivo:
                    break
        
        print(f"-> Se han recolectado {len(enlaces)} enlaces de iPhone 17 Pro Max ({cantidad_objetivo} objetivo)...")
        
        if len(enlaces) < cantidad_objetivo:
            try:
                boton_siguiente = driver.find_element(By.CSS_SELECTOR, "a.pagination__next")
                try:
                    driver.get(boton_siguiente.get_attribute("href"))
                except TimeoutException:
                    driver.execute_script("window.stop();")
                time.sleep(random.uniform(3, 5))
            except NoSuchElementException:
                print("No hay más páginas disponibles.")
                break

    return list(enlaces)[:cantidad_objetivo]

def extraer_detalle_ebay(driver, url, indice, total):
    """FASE 2: Ingresa a la página de detalle del iPhone y extrae las variables."""
    print(f"[{indice}/{total}] Extrayendo datos de: {url}")
    
    try:
        driver.get(url)
    except TimeoutException:
        print("   [!] Timeout en esta página, forzando lectura...")
        driver.execute_script("window.stop();")
    
    time.sleep(random.uniform(2, 4))
    
    datos = {
        "URL": url,
        "Titulo": "No encontrado",
        "Precio": "No encontrado",
        "Condicion": "No encontrada",
        "Vendedor": "No encontrado"
    }
    
    try:
        titulo_elem = driver.find_element(By.CSS_SELECTOR, "h1.x-item-title__mainTitle")
        datos["Titulo"] = titulo_elem.text
    except NoSuchElementException: pass

    try:
        precio_elem = driver.find_element(By.CSS_SELECTOR, "div.x-price-primary span.ux-textspans")
        datos["Precio"] = precio_elem.text
    except NoSuchElementException: pass

    try:
        condicion_elem = driver.find_element(By.CSS_SELECTOR, "div.x-item-condition-value span.ux-textspans")
        datos["Condicion"] = condicion_elem.text
    except NoSuchElementException: pass

    try:
        vendedor_elem = driver.find_element(By.CSS_SELECTOR, "div.x-sellercard-atf__info__about-seller span.ux-textspans")
        datos["Vendedor"] = vendedor_elem.text
    except NoSuchElementException: pass

    return datos

def main():
    driver = configurar_driver()
    datos_extraidos = []
    cantidad_a_extraer = 50
    
    try:
        print("--- INICIANDO FASE 1: RECOPILACIÓN DE LINKS (iPHONE 17 PRO MAX) ---")
        enlaces_productos = obtener_enlaces_ebay(driver, cantidad_objetivo=cantidad_a_extraer)
        
        if len(enlaces_productos) == 0:
            print("\n CRÍTICO: El bot no vio los enlaces. Asegúrate de presionar ENTER solo cuando cargue la página.")
            return 
            
        print(f"\n¡Se encontraron {len(enlaces_productos)} iPhones! Iniciando extracción...\n")
        
        print("--- INICIANDO FASE 2: EXTRACCIÓN DE DETALLES ---")
        for indice, url in enumerate(enlaces_productos, start=1):
            resultado = extraer_detalle_ebay(driver, url, indice, len(enlaces_productos))
            if resultado:
                datos_extraidos.append(resultado)
                
    except Exception as e:
        print(f"\nOcurrió un error inesperado: {e}")
                
    finally:
        driver.quit()
        print("\nNavegador cerrado de forma segura.")

    if datos_extraidos:
        df = pd.DataFrame(datos_extraidos)
        # Nombre del archivo de salida
        nombre_archivo = "dataset_iphone17_ebay.csv"
        df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
        print(f"\n¡TRABAJO TERMINADO! {len(datos_extraidos)} registros guardados exitosamente en '{nombre_archivo}'.")

if __name__ == "__main__":
    main()

--- INICIANDO FASE 1: RECOPILACIÓN DE LINKS (iPHONE 17 PRO MAX) ---
Abriendo la página de listado: https://www.ebay.com/sch/i.html?_nkw=iphone+17+pro+max

PAUSA DE SEGURIDAD ACTIVADA
1. Ve a la ventana de Chrome que se abrió.
2. Si eBay te pide verificar que eres humano, resuélvelo.
3. Asegúrate de estar viendo los iPhone 17 Pro Max en pantalla.
4. Vuelve a esta consola negra...


SEGUNDO BLOQUE DE CODIGO


In [ ]:
import pandas as pd


def clasificar_precio(precio):
    """
    Clasifica el precio según el rango esperado del iPhone 17 Pro Max.
    """
    if precio < 1199:
        return "🔴 ALERTA DE ESTAFA (Muy barato)"
    elif precio <= 1999:
        return "🟢 Precio normal"
    else:
        return "🟠 Precio muy alto"


def analizar_y_mostrar_tabla():
    # ===============================
    # Leer archivo
    # ===============================
    try:
        df = pd.read_csv("dataset_iphone17_ebay.csv")
    except FileNotFoundError:
        print("Error: No se encontró 'dataset_iphone17_ebay.csv'")
        return

    # ===============================
    # Eliminar registros incompletos
    # ===============================
    df = df[
        (df["Precio"] != "No encontrado") &
        (df["Titulo"] != "No encontrado")
    ].copy()

    # ===============================
    # Convertir precio a número
    # ===============================
    df["Precio_Numerico"] = (
        df["Precio"]
        .replace(r"[^\d.]", "", regex=True)
        .astype(float)
    )

    # ===============================
    # Filtrar solo iPhone 17 Pro Max
    # ===============================
    df = df[
        df["Titulo"].str.contains(
            "iphone 17 pro max",
            case=False,
            na=False
        )
    ].copy()

    # ===============================
    # Crear columna de alerta
    # ===============================
    df["Alerta"] = df["Precio_Numerico"].apply(clasificar_precio)

    # ===============================
    # Ordenar por precio
    # ===============================
    df = df.sort_values(
        by="Precio_Numerico",
        ascending=False
    )

    # ===============================
    # Seleccionar columnas finales
    # (Se elimina Condicion)
    # ===============================
    tabla_final = df[
        [
            "Titulo",
            "Precio_Numerico",
            "Alerta",
            "Vendedor",
            "URL"
        ]
    ].rename(
        columns={
            "Precio_Numerico": "Precio (USD)"
        }
    )

    # ===============================
    # Configuración de pantalla
    # ===============================
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 180)
    pd.set_option("display.max_colwidth", 55)

    # ===============================
    # Mostrar resultados
    # ===============================
    print("\n" + "=" * 140)
    print("ANÁLISIS DE MERCADO - iPHONE 17 PRO MAX")
    print("Rango esperado: USD 1,199 - USD 1,999")
    print("=" * 140)

    print(tabla_final.to_string(index=False))

    print("=" * 140)

    # ===============================
    # Resumen
    # ===============================
    total = len(tabla_final)
    normales = (tabla_final["Alerta"] == "🟢 Precio normal").sum()
    estafas = (tabla_final["Alerta"] == "🔴 ALERTA DE ESTAFA (Muy barato)").sum()
    altos = (tabla_final["Alerta"] == "🟠 Precio muy alto").sum()
    # ===============================
    # Guardar CSV
    # ===============================
    nombre_salida = "analisis_iphone17_tabla.csv"

    tabla_final.to_csv(
        nombre_salida,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\n✅ Archivo guardado como: {nombre_salida}")


if __name__ == "__main__":
    analizar_y_mostrar_tabla()


ANÁLISIS DE MERCADO - iPHONE 17 PRO MAX
Rango esperado: USD 1,199 - USD 1,999
                                                                                          Titulo  Precio (USD)                          Alerta             Vendedor                                   URL
       Apple IPHONE 17 Pro Max 1TB Naranja Cósmico A3526 Disponible inmediatamente 1000GB Nuevo-       4288.47               🟠 Precio muy alto Onlinehandel-McQueen https://www.ebay.com/itm/227380370313
           Apple IPHONE 17 Pro Max 256 GB Naranja Cósmico Disponible Inmediatamente A3526 Nuevo-       3236.18               🟠 Precio muy alto Onlinehandel-McQueen https://www.ebay.com/itm/227380370304
                 Apple IPHONE 17 Pro Max 256 GB Deep Blue A3526 inmediatamente disponible nuevo-       3230.18               🟠 Precio muy alto Onlinehandel-McQueen https://www.ebay.com/itm/227376308785
                 Apple iPhone 17 Pro Max 2 TB plata plata A3526 entrega inmediata 2000 GB NUEVO-       3149.00   

TERCER BLOQUE DE CODIGO


In [25]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.common.exceptions import TimeoutException

def configurar_driver():
    opciones = Options()
    opciones.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    opciones.add_argument("--disable-blink-features=AutomationControlled")
    opciones.add_argument("--start-maximized")
    
    servicio = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=servicio, options=opciones)
    driver.set_page_load_timeout(30) 
    return driver

def extraer_resenas_un_producto(driver, url, limite=20):
    print(f"Abriendo: {url}")
    
    try:
        driver.get(url)
    except TimeoutException:
        driver.execute_script("window.stop();")

    # Scroll para cargar contenido. Se ajusta si hay menos de 20 elementos cargados inicialmente
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(2)

    selectores = [
        "p.review-item-content", 
        "div.review-item-snippet", 
        "div.fdbk-item__comment-text", 
        "div.ux-seller-feedback-item__comment span",
        "div.fdbk-container__details__comment"
    ]
    
    lista_final = []
    
    for selector in selectores:
        elementos = driver.find_elements(By.CSS_SELECTOR, selector)
        for el in elementos:
            texto = el.text.strip()
            if texto and texto not in [item["Comentario"] for item in lista_final]:
                lista_final.append({"ID": len(lista_final) + 1, "Comentario": texto})
            
            if len(lista_final) >= limite:
                break
        if len(lista_final) >= limite:
            break

    return lista_final

def main():
    driver = None
    try:
        driver = configurar_driver()
        url_producto = "https://www.ebay.com/itm/287050959905"
        
        print("--- INICIANDO EXTRACCIÓN (Límite: 20) ---")
        datos = extraer_resenas_un_producto(driver, url_producto, limite=20)
        
        if datos:
            df = pd.DataFrame(datos)
            df.to_csv("resenas_20_items.csv", index=False, encoding='utf-8-sig')
            print(f"\n¡Éxito! {len(datos)} comentarios guardados en 'resenas_20_items.csv'.")
        else:
            print("\nNo se pudieron extraer suficientes comentarios.")
            
    except Exception as e:
        print(f"\nError: {e}")
    finally:
        if driver:
            driver.quit()

if __name__ == "__main__":
    main()

--- INICIANDO EXTRACCIÓN (Límite: 20) ---
Abriendo: https://www.ebay.com/itm/287050959905

¡Éxito! 20 comentarios guardados en 'resenas_20_items.csv'.


CUARTO BLOQUE DE CODIGO

In [35]:
import pandas as pd
from textblob import TextBlob

def clasificar_comentario(texto):
    """
    Analiza el texto localmente.
    - Criterio 1: Longitud (muy corto = probable bot).
    - Criterio 2: Subjetividad (cercana a 0 = objetivo/genérico = probable bot).
    """
    if pd.isna(texto):
        return "N/A"
    
    texto = str(texto)
    blob = TextBlob(texto)
    
    # Lógica de detección:
    # Si es muy corto (< 20 caracteres) O muy objetivo (subjetividad < 0.1)
    if len(texto) < 20 or blob.sentiment.subjectivity < 0.1:
        return "Bot (Sospechoso)"
    else:
        return "Humano (Detallado)"

def procesar_csv(archivo_entrada, archivo_salida):
    try:
        # 1. Cargar el archivo
        df = pd.read_csv(archivo_entrada)
        
        # 2. Asegurarse que existe la columna 'Comentario'
        # (Si tu archivo tiene otro nombre de columna, cámbialo aquí)
        columna = "Comentario" 
        
        if columna not in df.columns:
            print(f"Error: No se encontró la columna '{columna}' en el archivo.")
            return

        # 3. Aplicar el análisis
        print(f"Analizando {len(df)} comentarios...")
        df['Veredicto'] = df[columna].apply(clasificar_comentario)
        
        # 4. Guardar resultados
        df.to_csv(archivo_salida, index=False, encoding='utf-8-sig')
        print(f"Análisis completado. Resultados guardados en: {archivo_salida}")
        print(df[['Comentario', 'Veredicto']].head(10)) # Muestra los primeros 10
        
    except FileNotFoundError:
        print("Error: No se encuentra el archivo 'resenas_20_items.csv'. Asegúrate de que esté en la misma carpeta que este script.")

if __name__ == "__main__":
    procesar_csv('resenas_20_items.csv', 'analisis_resultado_final.csv')

Analizando 20 comentarios...
Análisis completado. Resultados guardados en: analisis_resultado_final.csv
                                               Comentario           Veredicto
0  Beautiful phone! Excellent, new as described and qu...  Humano (Detallado)
1  an A+++++++ seller. Item received in excellent pack...  Humano (Detallado)
2  Real Deal Ladies and gents….A++++ seller No import ...  Humano (Detallado)
3  ⭐️⭐️⭐️⭐️⭐️ Excellent seller. Fast shipping, item ex...  Humano (Detallado)
4  Love this purchase. Fast shipping and arrived here ...  Humano (Detallado)
5               thank you - as described- quickly arrived  Humano (Detallado)
6  Fast shipping and smooth transaction. Item was new ...  Humano (Detallado)
7  Item as described. Fast shipping. Perfect seller. T...  Humano (Detallado)
8                          Great seller ! Fast Shipping !  Humano (Detallado)
9  Once again SmartShop Hong Kong continued to be a fa...  Humano (Detallado)
